# Feature Importance Analysis (B)

## Objective
Determine which features contribute most to the classification of Seizure vs Healthy vs Inter-ictal signals.

## Method
We use a `RandomForestClassifier` to compute standard Gini importance (`feature_importances_`).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import os

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
data_path = '../data/processed/Bonn_EEG_Train.csv'

if not os.path.exists(data_path):
    print("Data file not found!")
else:
    df = pd.read_csv(data_path)
    print(f"Data loaded: {df.shape}")

## 2. Prepare X and y

In [ ]:
# Features
feature_cols = [
    'RMS', 'ZCR', 
    'Hjorth_Activity', 'Hjorth_Mobility', 'Hjorth_Complexity',
    'Envelope_Mean', 'Envelope_Max', 
    'Deriv1_Mean', 'Deriv1_Std', 
    'Deriv2_Mean', 'Deriv2_Std'
]

available_cols = [c for c in feature_cols if c in df.columns]
X = df[available_cols]

# Labels (Convert One-Hot to Class Index)
y_cols = [c for c in df.columns if c.startswith('y_')]
y = df[y_cols].idxmax(axis=1).apply(lambda s: int(s.split('_')[1]))

print("Features X shape:", X.shape)
print("Labels y shape:", y.shape)
print("Class distribution:\n", y.value_counts())

## 3. Train Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

## 4. Extract and Plot Importances

In [ ]:
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': available_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()